In [1]:
%pip install langchain-google-genai langchain-community langgraph duckduckgo-search

In [2]:
%pip install --upgrade --quiet  langchain-google-genai langchain-core

In [11]:
import os
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools import DuckDuckGoSearchRun
from langgraph.prebuilt import create_react_agent
from google.colab import userdata

In [4]:
os.environ['GOOGLE_API_KEY']=userdata.get('GOOGLE_API_KEY')

In [5]:
# 1. Initialize the Base LLM
model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite",api_key= os.getenv('GOOGLE_API_KEY'))


In [15]:
# 2. Invoke the bare LLM without any tools
resp=model.invoke("Who is the current prime minister of India and president of Americs")

In [16]:
# 3. Print the content of the response
print(resp.content)

[{'type': 'text', 'text': 'As of May 2024:\n\n*   **Prime Minister of India:** Narendra Modi\n*   **President of the United States:** Joe Biden', 'extras': {'signature': 'El4KXAERTTIPRXn6C6aNzbbcAfcAOSv/gsKbCtEsLzKX9f2xyzdp4Z711FxFOibfZp5pjP2DzmKux5kbsF7G0t8ruh0iEbhN/QF+RfsFDWhe82nXXh2uDcrlBYFqA2m+'}}]


In [7]:
%pip install ddgs

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 10.1 MB/s eta 0:00:00


In [8]:
# 1. Give the LLM access to the internet via DuckDuckGo
search_tool = DuckDuckGoSearchRun()

graph = create_react_agent(
    model,
    tools=[search_tool],
    prompt="You are a helpful assistant",
)



/tmp/ipykernel_2186/2112659766.py:4: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  graph = create_react_agent(


In [9]:
# 2. Create the agentic loop
resp=graph.invoke({"messages": [("user", "WHo is the president of America?")]})
# 3. Extract ONLY the final string output from the complex dictionary response
final_answer = resp['messages'][-1].content

print(f"Agent Answer:\n{final_answer}")

Agent Answer:
[{'type': 'text', 'text': 'The president of the United States is Donald Trump. He took office as the 47th president on January 20, 2025, having previously served as the 45th president from 2017 to 2021.', 'extras': {'signature': 'El4KXAERTTIP05JoGftQUzis4k+z+jvx/fScO+oprS2WGtqLryJjYnBayPU09hQNfgr/G5yA6qRjOKjZIkOome6IK/nh7dUUPaKFCHpu2cAoXvN4HD1XPqt/wWcZ+dOj'}}]


In [34]:
question='My name is John and I love cricket'
question2='What is my name and which sports I love to play?'
resp1=graph.invoke({"messages": [("user", question)]})
resp2=graph.invoke({"messages": [("user", question2)]})
final_answer1 = resp1['messages'][-1].content
final_answer2 = resp2['messages'][-1].content

print(f"Agent Answer:\n{final_answer1}")
print(f"Agent Answer:\n{final_answer2}")

Agent Answer:
[{'type': 'text', 'text': "Nice to meet you, John! It's great to have a fellow cricket fan here. \n\nWho is your favorite team or player? Are you currently following any specific matches or tournaments?", 'extras': {'signature': 'El4KXAERTTIPdkXgKpdOl40HjETv3AzneA3sLlpLdpGKVNN0zu+ykQQJ2Abc6TgvF3H7+MNq7TE0e6VarAGbac5bCfQxTnEgRZOz3itvrdhTIdMtbnuEkBBlWuZI6er0'}}]
Agent Answer:
[{'type': 'text', 'text': "I don't have access to your personal information, so I don't know your name or which sports you love to play! \n\nIf you tell me, I'll be happy to remember.", 'extras': {'signature': 'El4KXAERTTIPJ5gPFJ6MBmVMUCYgT4E5MoqlUfaS4n3cCHqeTU623iPreAef9mWqxcFMisjuL18ST3giObFIwFEz1n7XfTj/0ws3LuORB6Cu4vtYCBRN/fPZbpzz310e'}}]


In [10]:
# Write your agent builder script here
#Import MemorySaver from langgraph.checkpoint.memory.
#Create a memory object (memory = MemorySaver()).
#Re-create the agent (agent_with_memory = create_react_agent(...)), this time attaching the checkpointer=memory argument.
from langgraph.checkpoint.memory import MemorySaver
memory=MemorySaver()
agent_with_memory=create_react_agent(
    model,
    tools=[search_tool],
    checkpointer=memory,
    prompt="You are a helpful assistant",
)



/tmp/ipykernel_2186/1517953395.py:7: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_with_memory=create_react_agent(


In [15]:
question="Hi, my name is Alex and my favorite planet is Mars."
question2="What is my name, and which planet should I travel to based on my favorites?"
config = {
    "configurable": {
        "thread_id": "session-002",
        "user_id": "Prabh"
    }
}

resp1=agent_with_memory.invoke({"messages": [("user", question)]},config=config)
resp2=agent_with_memory.invoke({"messages": [("user", question2)]},config=config)
final_answer1 = resp1['messages'][-1].content
final_answer2 = resp2['messages'][-1].content

print(f"Agent Answer:\n{final_answer1}")
print(f"Agent Answer:\n{final_answer2}")

Agent Answer:
[{'type': 'text', 'text': "Hi Alex! It's nice to meet you. \n\nMars is a pretty awesome choice—the Red Planet has some incredible features, like Olympus Mons (the tallest volcano in the solar system) and Valles Marineris (a massive canyon system). \n\nHave you always been fascinated by space and Mars, or is there something specific you love about it?", 'extras': {'signature': 'El4KXAERTTIPCp1GOsmeMnU1IGuaIEITb7c+b1coG4B4bJ37hY7CCAh2Wu376y42vKVTqwrI0Dsw20f3jwMGI1zKWo/UyLdWyItpa4CuVVJ/Kw5AoF+Nq45ogZZ+vNmd'}}]
Agent Answer:
[{'type': 'text', 'text': "Your name is Alex, and based on your favorite, you should definitely travel to **Mars**! \n\nJust be sure to pack a heavy spacesuit, a rover for getting around, and maybe some potatoes (if you've seen *The Martian*)! Are you ready to book your ticket?", 'extras': {'signature': 'El4KXAERTTIPr4lODK82/O5J6qoivCB3rqQrDBcLzgCdFTB+vp5zIVC+0u0I5d/JwnfdNpCH4j2Adn4Ljf53A0AEW9ZPMlWvYDNavX0QdX71wzu68+HZ/+G+dGIXUHaY'}}]


In [16]:
%pip install -q Wikipedia


  Preparing metadata (setup.py) ... done


In [17]:
import pandas as pd
from langchain_core.tools import tool
from langchain_community.tools import WikipediaQueryRun
from langchain_community.utilities import WikipediaAPIWrapper
from langgraph.prebuilt import create_react_agent
import sqlite3


In [104]:
conn=sqlite3.connect(':memory:', check_same_thread=False)

In [103]:
df1=pd.read_csv('customerr.csv')
df2=pd.read_csv('sales2.csv')

In [105]:
df1.to_sql('customer', conn, if_exists='replace', index=False)
df2.to_sql('sales', conn, if_exists='replace', index=False)


200

In [113]:
cursor = conn.cursor()
for row in cursor.execute('SELECT * FROM customer  limit 10 '):
  print(row)


(1, 'Robert James', 'Lishire')
(2, 'Crystal Castillo', 'New Kimberlychester')
(3, 'Marcus Hayes', 'Smithbury')
(4, 'Christian Smith', 'Kelleyport')
(5, 'Olivia Vargas', 'Emilystad')
(6, 'Tyler Cruz', 'Lake Peterberg')
(7, 'Alyssa Smith', 'Joshuamouth')
(8, 'Kelly Barajas', 'South Alan')
(9, 'William Smith', 'South Justin')
(10, 'Adam Ortiz', 'South Tammy')


In [108]:
# It is time to build your custom tool! Notice the @tool decorator above the function.

# More importantly, look at the multi-line Docstring inside the function.
# This is the instruction manual for the AI. Because we provided the exact table schemas here,
# the LLM will know exactly how to write valid SQL to join them.

# Run this cell to register the tool.
@tool
def execute_sql(query: str) -> str:
    """The database contains customers (Customer_ID, Name, City)
        The database contains sales (Customer_ID, Purchase_Amount)
        Use this tool whenever the user asks for sales figures or customer data.
    """
    try:
        result = pd.read_sql_query(query, conn)
        return str(result.to_dict(orient='records'))
    except Exception as e:
        return f"Error executing query: {e}"



In [121]:
wikipedia=WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

In [122]:
tools = [execute_sql,wikipedia]
agent_smart_sql = create_react_agent(
    model,
    tools=tools,
)

# 6. Test it!
print("\nAgent is ready! Sending dual-tool request...\n")
response = agent_smart_sql.invoke({"messages": [("user", "what is our total sales revenue for Williamsfort city query the database for that? and what is percentages of ethnicities of people living in Chicago according to wikipedia")]})

print(f"Agent Answer:\n{response['messages'][-1].content}")



/tmp/ipykernel_2186/2974168830.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent_smart_sql = create_react_agent(



Agent is ready! Sending dual-tool request...

Agent Answer:
[{'type': 'text', 'text': 'Based on the database, the total sales revenue for **Williamsfort** is **$13.74**.\n\nAccording to Wikipedia (2020 Census data for the city of Chicago), the racial and ethnic breakdown of the population is as follows:\n\n* **White:** 35.9%\n* **Black or African American:** 29.2%\n* **Hispanic or Latino** (of any race): 29.8% *(Note: The remaining 70.2% belong to a non-Hispanic or Latino background)*\n* **Asian:** 7.0%\n* **Two or more races:** 10.8%\n* **Some other race:** 15.8%\n* **Native American or Alaska Native:** 0.1%', 'extras': {'signature': 'El4KXAERTTIP6dN3hMOhgMB6nXuTpnT9vAig7jh77J6SRxR+BfPvcElA1JdlCjdsoq4xsRNJHPfImVSfsPXUj+9RIwDjm4MOS76qDh7DWvkKvbuQvnSR8crqkBQUq9OD'}}]


Import TypedDict and Annotated from typing.
Import StateGraph, START, and END from langgraph.graph.
Import add_messages from langgraph.graph.message.
Import SystemMessage from langchain_core.messages.
Create a variable db_agent by calling create_react_agent(llm, tools=[execute_sql]).
Create a variable wiki_agent by calling create_react_agent(llm, tools=[wikipedia]).


In [123]:
from typing import TypedDict,Annotated
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langchain_core.messages import SystemMessage


In [124]:
db_agent=create_react_agent(model,tools=[execute_sql])
wiki_agent=create_react_agent(model,tools=[wikipedia])

/tmp/ipykernel_2186/3712428126.py:1: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  db_agent=create_react_agent(model,tools=[execute_sql])
/tmp/ipykernel_2186/3712428126.py:2: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  wiki_agent=create_react_agent(model,tools=[wikipedia])


Now we build the "Brain" of the operation: The Supervisor Node.

Write the script to:

Define a class AgentState(TypedDict) containing messages: Annotated[list, add_messages] and next_action: str.
Define a function supervisor_node(state: AgentState).
Inside the function, create a SystemMessage instructing the LLM to reply with EXACTLY the word "DATABASE" if the question is about sales/customers, or "WIKIPEDIA" if it is about general knowledge.
Call llm.invoke() passing the system message plus the state["messages"].
Write parsing logic to handle the output (whether the API returns a string or a list of dict blocks) and .strip().upper() the final text to get the decision.
Use an if/else block: If "DATABASE" is in the decision, return {"next_action": "database_expert"}. Otherwise, return "wikipedia_expert".


In [147]:
class AgentState(TypedDict):
  messages:Annotated[list,add_messages]
  next_action:str

def supervisor_node( state : AgentState):
  print("\n[SUPERVISOR] Analyzing user request...")
  sys_msg = SystemMessage(content="You are a supervisor router. Your job is to route the user's question to the correct expert.\n"
                                    "If the question is about internal sales, customers, or revenue, reply with EXACTLY the word 'DATABASE'.\n"
                                    "If the question is about general knowledge, history, or external facts, reply with EXACTLY the word 'WIKIPEDIA'.\n"
                                    "Only reply with one of those two words.")
  response=model.invoke([sys_msg]+state["messages"])
  content_str = response.content
  if isinstance(content_str, list):
      content_str = "".join([block.get("text", "") for block in content_str if isinstance(block, dict)])
  else:
      content_str = str(content_str)

  decision = content_str.strip().upper()

  if "DATABASE" in decision:
      return {"next_action": "database_expert"}
  else:
      return {"next_action": "wikipedia_expert"}



We have our Sub-Agents, but they don't know who they are yet. We must build wrapper nodes that dynamically inject a SystemMessage giving them a strict identity right before they are invoked.
     
Write the script to:          

Define db_node(state: AgentState). Inside, create a SystemMessage telling the agent it is a strict database analyst. Prepend this message to state["messages"], invoke db_agent, and return the last message.            
Define wiki_node(state: AgentState). Inside, create a SystemMessage telling the agent it is a general researcher. Prepend this message to state["messages"], invoke wiki_agent, and return the last message.
              

In [148]:
def db_node(state: AgentState):
    print("[SUPERVISOR] -> Routing task to the DATABASE Expert...")
    sys_msg = SystemMessage(content="you are a strict database analyst.")
    result = db_agent.invoke({"messages": [sys_msg] + state["messages"]})
    return {"messages": [result["messages"][-1]]}

def wiki_node(state: AgentState):
    print("[SUPERVISOR] -> Routing task to the WIKIPEDIA Expert...")
    sys_msg = SystemMessage(content="you are a general researcher.")
    result = wiki_agent.invoke({"messages": [sys_msg] + state["messages"]})
    return {"messages": [result["messages"][-1]]}


In [149]:
def route_edge(state: AgentState) -> str:
    return state["next_action"]

builder = StateGraph(AgentState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("database_expert", db_node)
builder.add_node("wikipedia_expert", wiki_node)

builder.add_edge(START, "supervisor")
builder.add_conditional_edges("supervisor", route_edge, {"database_expert": "database_expert", "wikipedia_expert": "wikipedia_expert"})
builder.add_edge("database_expert", END)
builder.add_edge("wikipedia_expert", END)

master_agent = builder.compile()



In [152]:
wikipedia=WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper())

In [153]:
# --- TEST IT ---
print("====================================")
response1 = master_agent.invoke({
    "messages": [("user", "What is the total sales revenue for customers living in Chicago?")]
})
print(f"\nFinal Output 1:\n{response1['messages'][-1].content}")

print("\n====================================")
response2 = master_agent.invoke({
    "messages": [("user", "Can you give me a brief history of the city of Chicago?")]
})
print(f"\nFinal Output 2:\n{response2['messages'][-1].content}")



[SUPERVISOR] Analyzing user request...
[SUPERVISOR] -> Routing task to the DATABASE Expert...

Final Output 1:
[{'type': 'text', 'text': 'The total sales revenue for customers living in Chicago is **$0.00** (there are no customers listed as living in Chicago in the database).', 'extras': {'signature': 'El4KXAERTTIPajG2J1f4Ua6UJHdaePAlOOH0ESwTP5+3raYPknIL509moqL8WtsW8KlWVkdKxD8rWBseKRY94XW89DUPCpd48bD7gvLtYgwXJHQY7u9siiRH9xiPw+J2'}}]


[SUPERVISOR] Analyzing user request...
[SUPERVISOR] -> Routing task to the WIKIPEDIA Expert...

Final Output 2:
[{'type': 'text', 'text': 'The history of Chicago is a remarkable tale of transformation, evolving from a swampy frontier outpost into one of the world\'s great economic and cultural metropolises in less than two centuries. \n\nHere is a brief overview of Chicago’s history, broken down by key eras:\n\n### 1. Pre-Colonial Era and Early Exploration\n* **Native Inhabitants:** Long before European contact, the area was inhabited by various Indigeno